In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_24_kinetic_monte_carlo.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 24: Kinetic Monte Carlo: Master Equation, BKL Algorithm, Poisson Clocks & Superbasins

_Session e8d9ef5e · Estimated time: ~120 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Explain the fundamental timescale separation between atomic vibrations ($10^{-14}\text{ s}$) and activated state transitions ($10^{-6}\text{--}10^3\text{ s}$) that motivates Kinetic Monte Carlo (KMC).
- Derive transition rates using Harmonic Transition State Theory (hTST) and the Arrhenius law $k = \nu_0 e^{-E_a / k_B T}$.
- Formulate the continuous-time discrete-state Master Equation and apply Trotter factorization to establish event independence.
- Implement the Bortz-Kalos-Lebowitz (BKL) / $N$-fold way / Gillespie algorithm with rejection-free event selection.
- Advance physical time using the Poisson random waiting-time distribution $\Delta t = -\ln(u) / R_{\text{total}}$.
- Solve the surface adsorption-desorption master equation analytically and verify convergence to the Langmuir isotherm.
- Diagnose the superbasin flickering trap problem and implement accelerated coarse-graining solutions.


In [2]:
ws.render_section('sec_kmc_motivation_and_tst')

### The Timescale Bottleneck of Molecular Dynamics

In classical Molecular Dynamics (MD), the numerical integration timestep is constrained by the highest-frequency vibrational modes (e.g., C-H stretches or Debye phonon frequencies) to $\Delta t \approx 10^{-15}\text{ s}$ (1 femtosecond). A simulation running for $10^7$ steps covers only $10\text{ ns}$ of physical time.

Yet many critical materials processes—including atomic surface diffusion, crystal epitaxial growth, grain boundary migration, void nucleation, and catalytic chemical reactions—involve **activated rare events** with energy barriers $E_a \approx 0.5\text{--}1.5\text{ eV}$. In MD, atoms spend $99.999999\%$ of CPU time executing uninteresting thermal oscillations around energy basin minima.

### Harmonic Transition State Theory (hTST)

By averaging out the fast vibrational oscillations, the system can be modeled as jumping between discrete potential energy basins. According to Vineyard's Harmonic Transition State Theory (hTST), the transition rate $k_{A \to B}$ from basin $A$ over a saddle point to basin $B$ is:

$$k_{A \to B} = \nu_0 \exp\left( -\frac{E_a}{k_B T} \right), \qquad \nu_0 = \frac{\prod_{i=1}^{3N} \nu_i^{(A)}}{\prod_{j=1}^{3N-1} \nu_j^{(\text{saddle})}}$$

where $E_a = E_{\text{saddle}} - E_A$ is the activation barrier, and $\nu_0$ is the **attempt frequency** (typically $\sim 10^{12}\text{--}10^{13}\text{ s}^{-1}$), given by the ratio of normal-mode vibrational frequencies at the basin minimum to those at the dividing saddle surface.

If $E_a = 0.6\text{ eV}$ at $T = 300\text{ K}$ ($k_B T \approx 0.02585\text{ eV}$):

$$k = 10^{13} \times e^{-0.6 / 0.02585} \approx 10^{13} \times 8.3 \times 10^{-11} \approx 830\text{ s}^{-1}$$

The average waiting time before a hop occurs is $\tau = 1/k \approx 1.2\text{ ms}$—requiring $10^{12}$ MD timesteps! Kinetic Monte Carlo (KMC) simulates these state-to-state hops directly, jumping across physical time milliseconds at a step.


In [3]:
ws.render_section('sec_master_equation_and_bkl')

### The Continuous-Time Master Equation

Let $P(s; t)$ be the probability that the system occupies discrete state $s$ at physical time $t$. Assuming a memoryless Markov process, the time evolution is governed by the **Master Equation**:

$$\frac{dP(s; t)}{dt} = \sum_{s'} \left[ W(s' \to s) P(s'; t) - W(s \to s') P(s; t) \right]$$

where $W(s \to s')$ is the transition rate (probability per unit time) from state $s$ to state $s'$. The first term represents probability flux *into* state $s$ (gain), while the second term represents flux *out of* state $s$ (loss).

### The BKL / $N$-Fold Way / Gillespie Algorithm

In 1975, Bortz, Kalos, and Lebowitz (BKL) introduced the $N$-fold way (independently developed for chemical reactions by Dan Gillespie in 1976). Rather than proposing trial moves and rejecting them (as in standard Metropolis MC), the BKL algorithm is **rejection-free**:

1. **Event Catalog:** From the current state $s$, identify all $M$ possible outbound transitions, each with rate $r_k = W(s \to s_k)$ for $k = 1, \dots, M$.
2. **Total Escape Rate:** Calculate the total escape rate from state $s$:

$$R_{\text{total}} = \sum_{k=1}^M r_k$$

3. **Event Selection:** Choose which event $i$ occurs with probability proportional to its rate:

$$P_i = \frac{r_i}{R_{\text{total}}}$$

Draw a uniform random number $u_1 \in (0, 1)$ and select event $i$ satisfying:

$$\sum_{k=1}^{i-1} r_k \le u_1 R_{\text{total}} < \sum_{k=1}^i r_k$$

4. **Advance State:** Execute event $i$ unconditionally: $s \leftarrow s_i$.


In [4]:
ws.render_section('sec_poisson_clock_and_sampling')

### The Poisson Waiting-Time Distribution

In a memoryless Markov process with constant total escape rate $R$, the probability $P_{\text{survival}}(t)$ that NO transition occurs during time interval $[0, t]$ satisfies:

$$\frac{dP_{\text{survival}}}{dt} = -R \, P_{\text{survival}}(t) \implies P_{\text{survival}}(t) = e^{-R t}$$

The probability density $p(\Delta t)$ that the first event occurs in the time interval $[\Delta t, \Delta t + d(\Delta t)]$ is:

$$p(\Delta t) = -\frac{dP_{\text{survival}}}{d(\Delta t)} = R e^{-R \Delta t}$$

This is an **exponential (Poisson) waiting-time distribution** with mean waiting time:

$$\langle \Delta t \rangle = \int_0^\infty \Delta t \, R e^{-R \Delta t} d(\Delta t) = \frac{1}{R}$$

### Inversion Method for Time Step Generation

To sample $\Delta t$ correctly from $p(\Delta t) = R e^{-R \Delta t}$, we equate the cumulative distribution function (CDF) to a uniform random deviate $u_2 \in (0, 1]$:

$$F(\Delta t) = \int_0^{\Delta t} R e^{-R t'} dt' = 1 - e^{-R \Delta t} = 1 - u_2$$

Solving for $\Delta t$ gives the stochastic time advance equation:

$$\Delta t = -\frac{\ln(u_2)}{R_{\text{total}}}$$

After every step, the global simulation clock advances by:

$$t_{\text{sim}} \leftarrow t_{\text{sim}} + \Delta t$$

Notice that when $R_{\text{total}}$ is small (high barriers), $\Delta t$ is huge, allowing KMC to bridge seconds or hours of physical time in seconds of CPU time!


In [5]:
ws.render_section('concept_bkl_event_selector_and_clock')

### Interactive Concept: BKL Event Selector and Poisson Clock

Explore the BKL event selection roulette and Poisson waiting-time clock.
Adjust activation energies ($E_1, E_2, E_3$), temperature $T$, and uniform random draw $u_1$ to see how the cumulative roulette selects transitions and how total rate $R$ governs time step $\Delta t$.

**Key Insight:** Lower barrier events exponentially dominate the total escape rate $R$. In the cumulative roulette wheel, each event occupies a segment proportional to $r_i / R$. A single random number $u_1$ deterministically selects the event, while a second independent random number $u_2$ sets the Poisson waiting time $\Delta t = -\ln(u_2)/R$.

In [6]:
ws.render_section('q24_01')

**Q1 · easy** · 💡

Why cannot standard Molecular Dynamics (MD) be used directly to simulate long-time kinetic processes like crystal growth or slow surface diffusion?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the ratio between vibrational periods ($10^{-14}\text{ s}$) and activation barrier crossing times ($10^{-3}\text{ s}$).

</div>
</details>

- [ ] **(A)** MD cannot simulate systems with more than 2 atoms.
- [ ] **(B)** High-frequency atomic vibrations constrain the MD timestep to ~1 femtosecond ($10^{-15}\text{ s}$), requiring billions of CPU steps to observe rare activated hops ($10^{-6}\text{--}10^3\text{ s}$).
- [ ] **(C)** Newton's laws of motion are fundamentally invalid for diffusion.
- [ ] **(D)** MD only works at $T = 0\text{ K}$.

In [7]:
ws.render_section('q24_02')

**Q2 · medium** · 💡

In Vineyard's Harmonic Transition State Theory (hTST), what determines the attempt frequency prefactor $\nu_0$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the vibrational normal-mode spectrum of the stable potential energy minimum with the non-imaginary normal modes of the transition saddle point.

</div>
</details>

- [ ] **(A)** The ratio of the product of all $3N$ vibrational normal-mode frequencies at the initial basin minimum to the product of the $3N-1$ stable vibrational frequencies at the transition saddle point.
- [ ] **(B)** The speed of light divided by the lattice constant.
- [ ] **(C)** The temperature divided by Planck's constant.
- [ ] **(D)** The mass of the electron.

In [8]:
ws.render_section('q24_03')

**Q3 · easy** · 💡

An atom hops across a surface with activation barrier $E_a = 0.50\text{ eV}$ and attempt frequency $\nu_0 = 10^{13}\text{ s}^{-1}$ at temperature $T = 500\text{ K}$ ($k_B T = 0.043087\text{ eV}$). According to transition state theory, calculate the transition rate $k$ in hops per second ($\text{s}^{-1}$) in scientific notation (enter the mantissa $M$ where $k = M \times 10^7$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the Arrhenius factor using the thermal energy $k_B T$ and scale by the vibrational attempt frequency.

</div>
</details>

_Numeric answer required._

In [9]:
ws.render_section('q24_04')

**Q4 · easy** · 💡

Why is the BKL ($N$-fold way / Gillespie) algorithm called 'rejection-free'?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Contrast BKL with standard Metropolis Monte Carlo where moves with $\Delta E > 0$ can be rejected.

</div>
</details>

- [ ] **(A)** Because it ignores energy conservation.
- [ ] **(B)** Because every simulation step selects and executes a valid event chosen from the rate catalog, so no proposed moves are ever rejected.
- [ ] **(C)** Because it accepts all moves with probability 0.5.
- [ ] **(D)** Because it only simulates exothermic reactions.

In [10]:
ws.render_section('q24_05')

**Q5 · medium** · 💡

In the Poisson waiting-time formula $\Delta t = -\frac{\ln u}{R_{\text{total}}}$, what does $u$ represent?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the transformation method for generating exponential random variables from uniform deviates.

</div>
</details>

- [ ] **(A)** The internal energy of the system
- [ ] **(B)** A uniform random number drawn on the interval $(0, 1]$
- [ ] **(C)** The velocity of the particle
- [ ] **(D)** The Planck constant

In [11]:
ws.render_section('q24_06')

**Q6 · medium** · 💡

From a current state, the total escape rate is $R_{\text{total}} = 5000\text{ s}^{-1}$. A uniform random number $u_2 = 0.367879$ (which is $1/e$) is drawn. Calculate the time step increment $\Delta t$ in seconds.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the inversion method for exponentially distributed waiting times given the total escape rate and uniform random deviate.

</div>
</details>

_Numeric answer required (in s)._

In [12]:
ws.render_section('q24_07')

**Q7 · medium** · 💡

In the concept cell 'BKL Event Selector', three competing pathways have barriers $E_1 = 0.45\text{ eV}$, $E_2 = 0.52\text{ eV}$, and $E_3 = 0.60\text{ eV}$ at $T = 500\text{ K}$. Which pathway will have the LARGEST probability of being selected?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how transition probability scales exponentially with activation barrier height under Arrhenius kinetics.

</div>
</details>

- [ ] **(A)** Pathway 1 (lowest barrier $E_1 = 0.45\text{ eV}$)
- [ ] **(B)** Pathway 2 ($E_2 = 0.52\text{ eV}$)
- [ ] **(C)** Pathway 3 (highest barrier $E_3 = 0.60\text{ eV}$)
- [ ] **(D)** All three pathways have identical probability

In [13]:
ws.render_section('q24_08')

**Q8 · medium** · 💡

What fundamental physical assumption underlies the Master Equation in Kinetic Monte Carlo?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Why does a particle 'forget' how it arrived in a potential energy basin before it hops again?

</div>
</details>

- [ ] **(A)** Quantum entanglement between all particles
- [ ] **(B)** The Markovian assumption: transitions between states depend only on the current state, because vibrational relaxation within a basin occurs much faster than the time between hops.
- [ ] **(C)** The assumption that all atoms move at the speed of sound
- [ ] **(D)** The requirement that potential energy must be zero everywhere

In [14]:
ws.render_section('q24_09')

**Q9 · medium** · 💡

Which of the following computational steps are executed in EACH iteration of a BKL KMC simulation? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the 4 steps of the BKL algorithm. Does KMC integrate Newton's equations?

</div>
</details>

- [ ] **(A)** Compute the total escape rate $R = \sum_k r_k$ from the current state.
- [ ] **(B)** Select an event $i$ using a uniform random number $u_1$ weighted by $r_i / R$.
- [ ] **(C)** Advance simulation time by $\Delta t = -\ln(u_2) / R$ using a second independent random number $u_2$.
- [ ] **(D)** Update the local environment and rate catalog for the affected sites.
- [ ] **(E)** Integrate Newton's equations of motion with a 1 fs Verlet timestep.

In [15]:
ws.render_section('q24_10')

**Q10 · medium** · 💡

A state has two outbound transitions with rates $r_1 = 300\text{ s}^{-1}$ and $r_2 = 700\text{ s}^{-1}$. What is the probability $P_1$ that transition 1 occurs?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Normalize the rate of the designated pathway by the total escape rate from the current state.

</div>
</details>

_Numeric answer required._

In [16]:
ws.render_section('sec_adsorption_desorption_kinetics')

### Surface Kinetics Model

Consider a surface with $N_{\text{sites}}$ identical adsorption sites exposed to a gas phase. Let $\theta(t) = N_{\text{occupied}}(t) / N_{\text{sites}}$ be the fractional surface coverage ($0 \le \theta \le 1$).

Two elementary processes compete:

1. **Adsorption:** Gas atoms hit empty sites at rate $r_A$ per site. The total adsorption rate is:

$$R_{\text{ads}} = r_A N_{\text{empty}} = r_A N_{\text{sites}} (1 - \theta)$$

2. **Desorption:** Adsorbed atoms overcome the binding energy and desorb at rate $r_D$ per occupied site. The total desorption rate is:

$$R_{\text{des}} = r_D N_{\text{occupied}} = r_D N_{\text{sites}} \theta$$

### The Mean-Field Master Equation

The deterministic macroscopic rate equation is:

$$\frac{d\theta(t)}{dt} = r_A [1 - \theta(t)] - r_D \theta(t) = r_A - (r_A + r_D)\theta(t)$$

Integrating from initial coverage $\theta(0) = \theta_0$ yields the exact relaxation trajectory:

$$\theta(t) = \theta_{\text{eq}} + (\theta_0 - \theta_{\text{eq}}) e^{-(r_A + r_D) t}$$

where the steady-state equilibrium coverage is the celebrated **Langmuir Isotherm**:

$$\theta_{\text{eq}} = \frac{r_A}{r_A + r_D} = \frac{K P}{1 + K P}$$

### Stochastic Fluctuations

In a finite system of $N_{\text{sites}}$, the instantaneous coverage fluctuates around $\theta_{\text{eq}}$. By the Central Limit Theorem, the variance of the steady-state coverage is:

$$\sigma_\theta^2 = \langle \theta^2 \rangle - \langle \theta \rangle^2 = \frac{\theta_{\text{eq}}(1 - \theta_{\text{eq}})}{N_{\text{sites}}}$$

KMC simulates the exact discrete birth-death master equation, reproducing both the mean relaxation curve and Poissonian fluctuations.


In [17]:
ws.render_section('concept_adsorption_desorption_kmc')

### Interactive Concept: Adsorption-Desorption KMC vs. Master Equation

Simulate surface adsorption-desorption kinetics using the BKL algorithm.
Adjust adsorption rate $r_A$, desorption rate $r_D$, number of lattice sites $N_{\text{sites}}$, and initial coverage $\theta_0$ to observe stochastic coverage trajectories approaching the Langmuir equilibrium.

**Key Insight:** The discrete stochastic KMC trajectory faithfully tracks the continuous analytical Master Equation solution $\theta(t) = \theta_{\text{eq}} + (\theta_0 - \theta_{\text{eq}})e^{-(r_A + r_D)t}$. As $N_{\text{sites}} \to \infty$, the relative amplitude of stochastic fluctuations vanishes as $1/\sqrt{N_{\text{sites}}}$, proving the convergence of KMC to deterministic continuum kinetics.

In [18]:
ws.render_section('q24_11')

**Q11 · easy** · 💡

In surface kinetics with adsorption rate $r_A = 1.5\text{ s}^{-1}$ and desorption rate $r_D = 0.5\text{ s}^{-1}$, calculate the steady-state equilibrium coverage $\theta_{\text{eq}}$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Set the net rate of coverage change to zero by balancing the rates of site adsorption and desorption.

</div>
</details>

_Numeric answer required._

In [19]:
ws.render_section('q24_12')

**Q12 · medium** · 💡

For the same system ($r_A = 1.5\text{ s}^{-1}, r_D = 0.5\text{ s}^{-1}$), what is the characteristic exponential relaxation time constant $\tau_{\text{relax}}$ in seconds?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the inverse of the sum of forward and reverse kinetic rates governing the approach to steady-state coverage.

</div>
</details>

_Numeric answer required (in s)._

In [20]:
ws.render_section('q24_13')

**Q13 · medium** · 💡

In the concept cell, how does increasing the number of surface sites $N_{\text{sites}}$ from 20 to 100 affect the coverage trajectory?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall how Central Limit Theorem fluctuations scale with system size $N$.

</div>
</details>

- [ ] **(A)** The equilibrium coverage $\theta_{\text{eq}}$ increases by a factor of 5.
- [ ] **(B)** The magnitude of stochastic fluctuations around the analytical curve decreases as $1/\sqrt{N_{\text{sites}}}$, making the trajectory smoother.
- [ ] **(C)** Desorption stops occurring completely.
- [ ] **(D)** The time constant $\tau_{\text{relax}}$ drops to zero.

In [21]:
ws.render_section('q24_14')

**Q14 · medium** · 💡

If the initial surface coverage is $\theta(0) = 0.10$ and the equilibrium coverage is $\theta_{\text{eq}} = 0.75$ with rate sum $r_A + r_D = 2.0\text{ s}^{-1}$, what is the coverage $\theta(t)$ at $t = 0.5\text{ s}$? (Recall $e^{-1} \approx 0.368$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Integrate the first-order linear relaxation differential equation for the coverage deficit relative to equilibrium.

</div>
</details>

- [ ] **(A)** $0.51$
- [ ] **(B)** $0.75$
- [ ] **(C)** $0.10$
- [ ] **(D)** $0.95$

In [22]:
ws.render_section('q24_15')

**Q15 · hard** · 💡

How is Trotter factorization used in the formal derivation of KMC from the continuous-time Markov operator?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Examine how operator splitting allows a composite continuous-time generator to be approximated as a sequence of independent local Poisson generators.

</div>
</details>

- [ ] **(A)** It decomposes the time evolution operator $e^{\mathcal{L} \Delta t} = \prod_k e^{\mathcal{L}_k \Delta t} + \mathcal{O}(\Delta t^2)$, proving that for small enough time increments, independent Poisson events can be executed sequentially without mutual interference.
- [ ] **(B)** It proves that KMC is identical to quantum electrodynamics.
- [ ] **(C)** It forces all transition rates to be equal.
- [ ] **(D)** It eliminates the need for random numbers.

In [23]:
ws.render_section('q24_16')

**Q16 · medium** · 💡

Which of the following data structures can be used to achieve $\mathcal{O}(\log M)$ or $\mathcal{O}(1)$ event selection in KMC with $M$ possible transitions? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Linear search across $M$ events takes $\mathcal{O}(M)$. How do advanced KMC codes accelerate roulette searches?

</div>
</details>

- [ ] **(A)** Binary search tree / cumulative sum tree ($\mathcal{O}(\log M)$ search and update)
- [ ] **(B)** Walker's Alias Method ($\mathcal{O}(1)$ event selection)
- [ ] **(C)** Binned rate groups / 24-tree (constant time per rate class)
- [ ] **(D)** Bubble sort after every step

In [24]:
ws.render_section('q24_17')

**Q17 · easy** · 💡

In the Langmuir adsorption model, what happens to the equilibrium coverage $\theta_{\text{eq}}$ if the gas pressure (and hence adsorption rate $r_A$) increases toward infinity while $r_D$ remains finite?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Analyze the steady-state site balance when the rate of arrival completely overwhelms the desorption rate.

</div>
</details>

- [ ] **(A)** $\theta_{\text{eq}} \to 0$
- [ ] **(B)** $\theta_{\text{eq}} \to 1$ (monolayer saturation)
- [ ] **(C)** $\theta_{\text{eq}} \to 0.5$
- [ ] **(D)** $\theta_{\text{eq}}$ oscillates wildly

In [25]:
ws.render_section('q24_18')

**Q18 · hard** · 💡

On a surface of $N_{\text{sites}} = 100$, the equilibrium coverage is $\theta_{\text{eq}} = 0.50$. Assuming independent site occupations under binomial statistics, calculate the expected standard deviation of fractional coverage fluctuations $\sigma_\theta$ to 3 decimal places.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the binomial variance formula for fractional site occupancy normalized by total site count.

</div>
</details>

_Numeric answer required._

In [26]:
ws.render_section('q24_19')

**Q19 · medium** · 💡

How does the transition rate $k$ change if the temperature $T$ is doubled from $300\text{ K}$ to $600\text{ K}$ for an activation barrier $E_a = 0.60\text{ eV}$? (Recall $k_B T_{300} \approx 0.02585\text{ eV}, k_B T_{600} \approx 0.0517\text{ eV}$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate how the exponential Arrhenius activation factor responds when the thermal denominator in the exponent is doubled.

</div>
</details>

- [ ] **(A)** It increases by exactly a factor of 2.
- [ ] **(B)** It increases by an enormous factor of $\sim 10^5$, because the Arrhenius factor changes from $e^{-23.2} \approx 8.3 \times 10^{-11}$ to $e^{-11.6} \approx 9.1 \times 10^{-6}$.
- [ ] **(C)** It decreases by a factor of 2.
- [ ] **(D)** It remains strictly constant.

In [27]:
ws.render_section('q24_20')

**Q20 · easy** · 💡

If the total escape rate from a state is $R = 10^6\text{ s}^{-1}$, what is the average waiting time $\langle \Delta t \rangle$ before an event occurs?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the first moment of an exponential Poisson distribution to the total escape frequency.

</div>
</details>

- [ ] **(A)** $1\text{ microsecond } (10^{-6}\text{ s})$
- [ ] **(B)** $1\text{ millisecond } (10^{-3}\text{ s})$
- [ ] **(C)** $1\text{ second}$
- [ ] **(D)** $1\text{ femtosecond } (10^{-15}\text{ s})$

In [28]:
ws.render_section('sec_superbasin_and_flickering_bottleneck')

### The Low-Barrier / Flickering Trap Problem

The standard BKL algorithm works brilliantly when all accessible barriers are of comparable height. However, materials systems often feature a **hierarchy of energy barriers**.

Consider an adatom hopping between two adjacent surface sites $A$ and $B$ separated by a very low barrier $E_{\text{flicker}} = 0.15\text{ eV}$, while all escape pathways to the rest of the surface require crossing a high barrier $E_{\text{escape}} = 0.70\text{ eV}$:

- The fast hopping rate between $A$ and $B$ is:

$$r_{\text{fast}} = \nu_0 e^{-0.15 / k_B T} \approx 10^{13} e^{-3.48} \approx 3.1 \times 10^{11}\text{ s}^{-1}$$

- The slow escape rate is:

$$r_{\text{slow}} = \nu_0 e^{-0.70 / k_B T} \approx 10^{13} e^{-16.24} \approx 8.8 \times 10^5\text{ s}^{-1}$$

The ratio of rates is $r_{\text{fast}} / r_{\text{slow}} \approx 350,000$! In standard KMC, the system will jump back and forth between $A$ and $B$ (flicker) roughly $350,000$ times before escaping. Each flickering step advances the physical clock by only:

$$\Delta t \sim \frac{1}{r_{\text{fast}}} \approx 3.2\times 10^{-12}\text{ s}$$

The simulation is trapped in a **superbasin**: it burns millions of CPU cycles executing trivial back-and-forth oscillations, and physical time progression grinds to a complete standstill!

### Solutions: Coarse-Graining and Absorbing Markov Chains

Several sophisticated methods overcome this bottleneck:

1. **Basin-Hopping & Superbasin KMC:** Detect fast-equilibrating sets of states (superbasins). Treat the entire superbasin as a single coarse-grained macro-state with internal Boltzmann equilibrium $P(i) = e^{-\beta E_i} / Z_{\text{basin}}$. The effective escape rate from the superbasin to state $C$ is:

$$R_{\text{basin} \to C} = \sum_{i \in \text{basin}} P(i) r_{i \to C}$$

2. **First-Passage Time Distribution / Absorbing Markov Chains:** Mathematically integrate out the fast internal transitions by computing the exact first-passage probability distribution for escaping the basin, leaping out of the superbasin in a single step with a correctly sampled large time increment $\Delta t_{\text{escape}}$.


In [29]:
ws.render_section('concept_superbasin_flickering_trap')

### Interactive Concept: The Superbasin Flickering Trap

Diagnose the KMC low-barrier trap and compare flickering vs. Escape rates.
Adjust the flickering barrier $E_{\text{flicker}}$, escape barrier $E_{\text{escape}}$, and temperature $T$ to observe the energy landscape and the distribution of escape times.

**Key Insight:** When $E_{\text{flicker}} \ll E_{\text{escape}}$, the number of wasted flickering hops scales exponentially as $N_{\text{flicker}} \sim e^{(E_{\text{escape}} - E_{\text{flicker}})/k_B T}$. Detecting superbasins and coarse-graining fast internal transitions restores KMC time progression without sacrificing kinetic accuracy.

In [30]:
ws.render_section('q24_21')

**Q21 · medium** · 💡

What causes the 'flickering trap' or 'superbasin problem' in standard Kinetic Monte Carlo?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the consequence for simulation efficiency when the total escape rate is dominated by a rapid, reversible low-barrier process.

</div>
</details>

- [ ] **(A)** The computer runs out of random numbers.
- [ ] **(B)** A few states are separated by very low energy barriers ($E_{\text{low}} \ll E_{\text{escape}}$), causing the simulation to execute millions of rapid back-and-forth hops with tiny time increments $\Delta t \sim 1/r_{\text{fast}}$, stalling physical time advance.
- [ ] **(C)** The temperature reaches absolute zero.
- [ ] **(D)** The master equation fails to conserve total probability.

In [31]:
ws.render_section('q24_22')

**Q22 · hard** · 💡

At $T = 500\text{ K}$ ($k_B T = 0.043087\text{ eV}$), a dimer has a flickering barrier $E_{\text{flicker}} = 0.20\text{ eV}$ and an escape barrier $E_{\text{escape}} = 0.70\text{ eV}$. Assuming equal attempt frequencies, what is the expected number of flickering hops before escaping the trap? (Enter to nearest 1000).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the ratio of the fast intra-basin rate to the slow basin escape rate governed by the barrier difference.

</div>
</details>

_Numeric answer required._

In [32]:
ws.render_section('q24_23')

**Q23 · medium** · 💡

How does coarse-grained superbasin KMC solve the flickering bottleneck?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How can states connected by fast reversible transitions be treated collectively?

</div>
</details>

- [ ] **(A)** By setting the fast rate to zero.
- [ ] **(B)** By grouping fast-equilibrating states into a single macro-state, assuming internal quasi-equilibrium, and computing an effective escape rate from the macro-state to outer states.
- [ ] **(C)** By lowering the temperature to freeze the system.
- [ ] **(D)** By converting the simulation to molecular dynamics.

In [33]:
ws.render_section('q24_24')

**Q24 · easy** · 💡

What happens to the ratio of the intra-trap flickering rate to the escape rate $r_{\text{flicker}} / r_{\text{escape}}$ as temperature $T$ is decreased from 800 K to 300 K?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the Arrhenius ratio for two competing processes with differing activation barriers as thermal energy decreases.

</div>
</details>

- [ ] **(A)** The ratio drops to 1.
- [ ] **(B)** The ratio grows exponentially, making the flickering trap vastly worse at lower temperatures.
- [ ] **(C)** The ratio becomes negative.
- [ ] **(D)** Both rates become identical.

In [34]:
ws.render_section('q24_25')

**Q25 · medium** · 💡

Which of the following physical applications are well suited for simulation by Kinetic Monte Carlo? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Remember: KMC coarse-grains out fast vibrations to model rare-event state-to-state hops.

</div>
</details>

- [ ] **(A)** Thin-film epitaxial growth (deposition, surface diffusion, island nucleation)
- [ ] **(B)** Heterogeneous catalysis and surface reaction networks (e.g. CO oxidation on Pt)
- [ ] **(C)** Defect diffusion and radiation damage annealing in nuclear reactor cladding
- [ ] **(D)** High-frequency femtosecond vibrational spectroscopy of water molecules

In [35]:
ws.render_section('q24_26')

**Q26 · medium** · 💡

What is an 'off-lattice' or 'adaptive' Kinetic Monte Carlo (AKMC) simulation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How do you apply KMC to amorphous solids or complex defect geometries where no fixed crystal lattice exists?

</div>
</details>

- [ ] **(A)** A simulation where particles are restricted to rigid cubic lattice points.
- [ ] **(B)** A simulation where atomic coordinates are continuous in space, and transition pathways and saddle points are discovered on-the-fly using saddle-search methods (like the Dimer method or NEB) without pre-defined lattice catalogs.
- [ ] **(C)** A simulation that does not use a computer.
- [ ] **(D)** A Monte Carlo simulation that runs backward in time.

In [36]:
ws.render_section('q24_27')

**Q27 · hard** · 💡

In calculating activation barriers $E_a$ for KMC rate catalogs using density functional theory (DFT) or empirical potentials, what algorithm is most widely used to find the minimum energy path (MEP) and saddle point between two known states?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What method connects initial and final states with a chain of images tethered by spring forces?

</div>
</details>

- [ ] **(A)** Nudged Elastic Band (NEB)
- [ ] **(B)** Simulated Annealing
- [ ] **(C)** Metropolis spin flip
- [ ] **(D)** Steepest descent energy minimization

In [37]:
ws.render_section('q24_28')

**Q28 · medium** · 💡

In a KMC simulation, 100,000 steps were executed. If the average escape rate across all steps was $\langle R \rangle = 200\text{ s}^{-1}$, what is the total simulated physical time in seconds?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the total elapsed physical time to the cumulative sum of expected increments for the given average transition rate.

</div>
</details>

_Numeric answer required (in s)._

In [38]:
ws.render_section('q24_29')

**Q29 · medium** · 💡

Why is it mathematically INCORRECT to simply advance the simulation clock by the deterministic mean $\Delta t = 1/R_{\text{total}}$ at every step instead of drawing from the exponential distribution $\Delta t = -\ln(u)/R$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider whether waiting times in real Poisson decay processes are deterministic or stochastic.

</div>
</details>

- [ ] **(A)** Using a fixed $\Delta t = 1/R$ correctly models the first moment, but eliminates all temporal variance and higher moments of the Poisson point process, distorting correlation functions and fluctuations in multi-step kinetics.
- [ ] **(B)** Because $1/R$ is always negative.
- [ ] **(C)** Because $\Delta t$ must always be zero in Monte Carlo.
- [ ] **(D)** Because the computer cannot divide by $R$.

In [39]:
ws.render_section('q24_30')

**Q30 · hard** · 💡

In an absorbing Markov chain formulation of superbasin escape, let $\mathbf{Q}$ be the substochastic transition matrix between transient states within the basin. What mathematical matrix $\mathbf{N}$ yields the expected number of times each state is visited before escape?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Sum the powers of the transient transition matrix representing the probability of remaining within the basin after successive steps.

</div>
</details>

- [ ] **(A)** The fundamental matrix $\mathbf{N} = (\mathbf{I} - \mathbf{Q})^{-1}$
- [ ] **(B)** $\mathbf{N} = \mathbf{Q}^2$
- [ ] **(C)** $\mathbf{N} = \mathbf{I} + \mathbf{Q}$
- [ ] **(D)** $\mathbf{N} = \det(\mathbf{Q}) \mathbf{I}$

In [40]:
ws.render_section('q24_31')

**Q31 · medium** · 💡

In a lattice KMC simulation of vacancy diffusion in a crystal, when a vacancy swaps with a neighboring atom, which transition rates need to be updated in the catalog?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Are short-range interatomic potentials affected by moves that occur far away?

</div>
</details>

- [ ] **(A)** Only the rates of the local sites in the immediate coordination neighborhood of the moving atom and vacancy.
- [ ] **(B)** All $10^6$ rates across the entire crystal must be completely recomputed.
- [ ] **(C)** No rates ever need to be updated.
- [ ] **(D)** Only the rates of the boundary atoms.

In [41]:
ws.render_section('q24_32')

**Q32 · easy** · 💡

If a surface starts fully empty ($\theta_0 = 0$), with adsorption rate $r_A = 2.0\text{ s}^{-1}$ and zero desorption ($r_D = 0.0\text{ s}^{-1}$, irreversible adsorption), what is the fractional coverage $\theta(t)$ at $t = 0.5\text{ s}$? (Recall $e^{-1} \approx 0.368$).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Integrate the rate equation for irreversible first-order site filling over the elapsed time interval.

</div>
</details>

_Numeric answer required._

In [42]:
ws.render_section('q24_33')

**Q33 · medium** · 💡

In Dan Gillespie's formulation of the Stochastic Simulation Algorithm (SSA) for chemical reaction networks, what is the 'propensity function' $a_j(\mathbf{x})$ of reaction $j$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice that Gillespie's propensity $a_j$ is the chemical kinetics equivalent of the KMC transition rate $r_k$.

</div>
</details>

- [ ] **(A)** The probability per unit time that reaction $j$ occurs, given the current population state $\mathbf{x}$.
- [ ] **(B)** The equilibrium constant of reaction $j$.
- [ ] **(C)** The enthalpy change of reaction $j$.
- [ ] **(D)** The activation energy in Joules.

In [43]:
ws.render_section('q24_34')

**Q34 · hard** · 💡

How does detailed balance relate forward and reverse transition rates $W(A \to B)$ and $W(B \to A)$ in an equilibrium KMC simulation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Equate the microscopic probability flux between two configurations using their canonical Boltzmann equilibrium weights.

</div>
</details>

- [ ] **(A)** $\frac{W(A \to B)}{W(B \to A)} = \frac{e^{-\beta E_B}}{e^{-\beta E_A}} = e^{-\beta(E_B - E_A)}$
- [ ] **(B)** $W(A \to B) = W(B \to A)$ always
- [ ] **(C)** $W(A \to B) + W(B \to A) = 1$
- [ ] **(D)** $\frac{W(A \to B)}{W(B \to A)} = 0$

In [44]:
ws.render_section('q24_35')

**Q35 · medium** · 💡

Two states $A$ and $B$ have energies $E_A = 0.0\text{ eV}$ and $E_B = 0.10\text{ eV}$ at $T = 300\text{ K}$ ($k_B T = 0.02585\text{ eV}$). If the forward transition rate is $W(A \to B) = 100\text{ s}^{-1}$, calculate the reverse transition rate $W(B \to A)$ in $\text{s}^{-1}$ to the nearest integer.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the condition of detailed balance using the energy difference between states $A$ and $B$.

</div>
</details>

_Numeric answer required (in s^-1)._

In [45]:
ws.render_section('q24_36')

**Q36 · medium** · 💡

Why do adatoms diffusing on close-packed metal surfaces (like FCC (111)) often hop via 'exchange' mechanisms rather than simple over-the-bridge hopping?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall concerted exchange diffusion on metal surfaces discovered by Feibelman (1990).

</div>
</details>

- [ ] **(A)** Because on many metals (e.g. Al, Pt), pushing a substrate atom out while the adatom takes its place has a lower activation barrier than climbing over the top of substrate atoms.
- [ ] **(B)** Because the bridge site has infinite potential energy.
- [ ] **(C)** Because the exchange mechanism violates Newton's laws.
- [ ] **(D)** Because adatoms can only move in diagonal directions.

In [46]:
ws.render_section('q24_37')

**Q37 · medium** · 💡

What happens if a rate catalog in a KMC simulation inadvertently omits a physically accessible low-barrier transition pathway?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Can a rejection-free KMC simulation choose an event that is not in its catalog?

</div>
</details>

- [ ] **(A)** The simulation produces physically incorrect kinetics and may become artificially trapped in metastable structures that would naturally decay via the omitted pathway.
- [ ] **(B)** The simulation automatically finds the missing pathway using quantum tunneling.
- [ ] **(C)** The total probability exceeds 1.
- [ ] **(D)** The timestep $\Delta t$ becomes negative.

In [47]:
ws.render_section('q24_38')

**Q38 · medium** · 💡

In transition state theory, if the attempt frequency is $\nu_0 = 5.0 \times 10^{12}\text{ s}^{-1}$ and the activation barrier is $E_a = 0.30\text{ eV}$ at $T = 300\text{ K}$ ($k_B T = 0.02585\text{ eV}$), what is the mean waiting time $\tau$ for this transition in microseconds ($\mu\text{s}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the transition rate from the Arrhenius relation, invert to obtain the mean waiting time, and convert to microsecond units.

</div>
</details>

_Numeric answer required (in microseconds)._

In [48]:
ws.render_section('q24_39')

**Q39 · medium** · 💡

Which of the following methods are effective strategies to accelerate KMC when confronted with low-barrier superbasins? (Select all that apply)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look for algorithms that eliminate fast internal hops while preserving relative exit probabilities.

</div>
</details>

- [ ] **(A)** Coarse-graining the superbasin into a single macro-state with effective exit rates
- [ ] **(B)** Absorbing Markov Chain / First-Passage Time algorithms that leap out of the basin in a single step
- [ ] **(C)** Increasing the barrier height of the fast transition artificially (with appropriate rate rescaling)
- [ ] **(D)** Replacing random numbers with the constant 0.5

In [49]:
ws.render_section('q24_40')

**Q40 · medium** · 💡

In a KMC simulation with 10 competing processes, why can a single random number $u_1 \in (0, 1)$ select the event regardless of how many processes exist?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think of partitioning a line of length $R_{\text{total}}$ into segments of length $r_k$ and throwing a dart at location $u_1 R_{\text{total}}$.

</div>
</details>

- [ ] **(A)** Because the cumulative rate array $\sum_{k=1}^i r_k$ maps the entire interval $[0, R_{\text{total}})$ into $M$ non-overlapping contiguous bins, each of length $r_i$.
- [ ] **(B)** Because only one process can have a non-zero rate.
- [ ] **(C)** Because all processes have equal probability.
- [ ] **(D)** Because random numbers are 64-bit integers.

In [50]:
ws.render_section('q24_41')

**Q41 · hard** · 💡

In Dan Gillespie's Tau-Leaping method (an approximate accelerated KMC algorithm), how are multiple fast events handled?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review the concept of Tau-Leaping as a bridge between exact discrete KMC and continuous chemical Langevin equations.

</div>
</details>

- [ ] **(A)** The simulation leaps forward by a pre-selected time step $\tau$, and the number of firings of each reaction channel during $\tau$ is sampled as an independent Poisson random variable $\mathcal{P}(r_k \tau)$, assuming propensities remain approximately constant during $\tau$.
- [ ] **(B)** All reactions are stopped completely.
- [ ] **(C)** The system switches to quantum Monte Carlo.
- [ ] **(D)** Only the slowest reaction is allowed to fire.

In [51]:
ws.render_section('q24_42')

**Q42 · easy** · 💡

A radioactive decay or atomic desorption process has a single transition channel with rate $r = 10.0\text{ s}^{-1}$ ($R_{\text{total}} = 10.0\text{ s}^{-1}$). If the uniform random deviate drawn is $u = 0.50$, calculate the sampled Poisson waiting time $\Delta t$ in seconds to 3 decimal places.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the standard exponential random variate inversion formula using the given random deviate and escape rate.

</div>
</details>

_Numeric answer required (in s)._

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()